# PACS datasets and transforms
Written only; no cells executed. See task2/README.md for execution order.

In [ ]:
from pathlib import Path
import json
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

def image_transform(training):
    operations = [transforms.Resize((256, 256))]
    operations += ([transforms.RandomCrop(224), transforms.RandomHorizontalFlip()]
                   if training else [transforms.CenterCrop(224)])
    return transforms.Compose(operations + [transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


class PACSSource(Dataset):
    def __init__(self, root, records, training=False):
        self.root, self.records = Path(root), records
        self.transform = image_transform(training)
        if any(Path(r['path']).parts[0] not in SOURCES for r in records):
            raise ValueError('Source dataset cannot include target images.')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        with Image.open(self.root / record['path']) as im:
            image = self.transform(im.convert('RGB'))
        return image, record['label']

class UnlabeledSketch(Dataset):
    """Scan images without constructing class labels or stratifying target data.

    Decoding failures are logged before training. Crops/flips use a separate CPU
    RNG stream, so adding target transforms cannot change the source augmentation
    sequence relative to ERM. Use with num_workers=0.
    """
    def __init__(self, root, output_dir, seed):
        self.root = Path(root)
        folder = self.root / 'sketch'
        if not folder.is_dir():
            raise FileNotFoundError(f'Missing target directory: {folder}')
        candidates = sorted(p for p in folder.rglob('*')
                            if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'})
        self.paths, rejected = [], []
        for path in candidates:
            try:
                with Image.open(path) as im:
                    im.convert('RGB').load()
                self.paths.append(path)
            except (OSError, ValueError) as exc:
                rejected.append(dict(path=path.relative_to(self.root).as_posix(), reason=str(exc)))
        if not self.paths:
            raise ValueError('No decodable Sketch images found.')
        manifest = dict(accepted=[p.relative_to(self.root).as_posix() for p in self.paths],
                        rejected=rejected, rule='Exclude only images that cannot be decoded as RGB.')
        output = Path(output_dir) / 'target_manifest.json'
        if output.exists() and json.loads(output.read_text(encoding='utf-8')) != manifest:
            raise ValueError('Target manifest changed; use a new output directory.')
        output.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
        print(f'Unlabeled Sketch: {len(self.paths)} usable images, {len(rejected)} decoding failures.')
        self.transform = image_transform(training=True)
        self.rng_state = torch.Generator().manual_seed(seed + 4).get_state()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as im:
            rgb = im.convert('RGB')
        with torch.random.fork_rng(devices=[]):
            torch.set_rng_state(self.rng_state)
            image = self.transform(rgb)
            self.rng_state = torch.get_rng_state()
        return image, -1


